# GitLab Scraping – Blinde Signaturen


In [1]:
import os
import time
import requests
import pandas as pd

from dotenv import load_dotenv
from tqdm import tqdm
from urllib.parse import quote

load_dotenv()

GITLAB_TOKEN = os.getenv("GITLAB_TOKEN")  # optional

HEADERS = {}
if GITLAB_TOKEN:
    HEADERS["PRIVATE-TOKEN"] = GITLAB_TOKEN

API_BASE = "https://gitlab.com/api/v4"

# GitLab-Suche durchsucht name, description und path (NICHT das README).
SEARCH_TERMS = [
    '"blind signatures"',
    '"anonymous credentials"',
    '"blind"',
    '"blind RSA"',
    '"blind token"',

    '"partially blind signature"',
    '"blind schnorr"',
    '"blind BLS"',                      #Boneh-Lynn-Shacham
    '"VOPRF"',
    '"OPRF" "token"',
    '"RSA blind signatures"',

    '"anonymous tokens"',
    '"private access token"',
    '"trust token"',
    '"private state token"',
    '"unlinkable token"',
    '"challenge bypass"',

    '"blinded message"',
    '"blind issuance"',
]

# Dieselben Pflichtbegriffe wie beim GitHub-Scraping
REQUIRED_TERMS = [
    "blind signature",
    "blind signatures",
    "blind signing",
    "blind sign",
    "blind rsa",
    "blind schnorr",
    "blinded message",
    "blinding factor",
    "blind issuance",
    "unblind",
    "partially blind",
    "voprf",
    "anonymous token",
    "private access token",
    "chaumian",
]

GITHUB_FILE  = "blind_signature_repos_v2_final.xlsx"   # bestehende GitHub-Ergebnisse
OUTPUT_FILE  = "gitlab_only_repos_v2.xlsx"
MIN_STARS    = 0   # GitLab-Projekte haben generell weniger Stars als GitHub


In [2]:
def search_projects(query, per_page=50, max_pages=10):
    """Sucht GitLab-Projekte über die Projects-API (name/description/path)."""
    all_items = []

    for page in range(1, max_pages + 1):
        params = {
            "search": query,
            "order_by": "star_count",
            "sort": "desc",
            "per_page": per_page,
            "page": page,
        }
        response = requests.get(f"{API_BASE}/projects", headers=HEADERS, params=params)

        if response.status_code != 200:
            print("Error:", response.status_code, response.text[:200])
            break

        items = response.json()
        if not items:
            break

        all_items.extend(items)

        # Weniger Ergebnisse als per_page → letzte Seite erreicht
        if len(items) < per_page:
            break

        time.sleep(0.5)

    return all_items


def get_readme(project_id, default_branch):
    candidates = ["README.md", "README.rst", "README.txt", "README", "readme.md"] #different data types and names

    for filename in candidates:
        url = f"{API_BASE}/projects/{project_id}/repository/files/{quote(filename, safe='')}/raw"
        params = {"ref": default_branch or "main"}
        response = requests.get(url, headers=HEADERS, params=params)

        if response.status_code == 200:
            return response.text

    return ""


def is_relevant(readme_text):
    """True wenn mindestens ein Pflichtbegriff im README vorkommt."""
    t = readme_text.lower()
    return any(term in t for term in REQUIRED_TERMS)


def classify_readme(text):
    t = text.lower()
    found = [term for term in REQUIRED_TERMS if term in t]
    return {
        "matched_terms":          ", ".join(found),
        "mentions_anti_bot":      any(k in t for k in ["captcha", "bot detection", "challenge bypass", "trust token", "private state token"]),
        "mentions_fraud_prevent": any(k in t for k in ["fraud prevention", "fraud detection", "private reporting", "touchpoint"]),
        "mentions_credentials":   any(k in t for k in ["anonymous credential", "anoncred", "de-identified", "selective disclosure"]),
        "mentions_advertising":   any(k in t for k in ["adveil", "ad measurement", "privacy-preserving ad", "anonymous ad"]),
        "mentions_privacy":       "privacy" in t,
        "mentions_authentication": "authentication" in t,
    }


In [3]:
# ── GitLab durchsuchen ───────────────────────────────────────────────────────

seen_ids = set()
all_rows = []
skipped = 0

for term in SEARCH_TERMS:
    projects = search_projects(term)

    for proj in tqdm(projects, desc=term):
        pid = proj["id"]

        if pid in seen_ids:
            continue
        seen_ids.add(pid)

        if proj.get("star_count", 0) < MIN_STARS:
            continue

        readme = get_readme(pid, proj.get("default_branch"))

        if not is_relevant(readme):
            skipped += 1
            time.sleep(0.5)
            continue

        analysis = classify_readme(readme)

        row = {
            "search_term": term,
            "repo_name":   proj["path"],
            "owner":       proj["namespace"]["path"] if proj.get("namespace") else "",
            "stars":       proj.get("star_count", 0),
            "forks":       proj.get("forks_count", 0),
            "created_at":  proj.get("created_at", ""),
            "updated_at":  proj.get("last_activity_at", ""),
            "description": proj.get("description", ""),
            "topics":      ", ".join(proj.get("topics", [])),
            "repo_url":    proj["web_url"],
            "platform":    "gitlab",
            "readme":      readme[:5000],
        }

        row.update(analysis)
        all_rows.append(row)
        time.sleep(0.5)

df_gitlab = pd.DataFrame(all_rows)
print(f"\nGitLab: {len(df_gitlab)} relevante Repos gefunden, {skipped} gefiltert")

"blind RSA": 100%|██████████| 4/4 [00:02<00:00,  1.84it/s]
"blind token": 0it [00:00, ?it/s]
"partially blind signature": 0it [00:00, ?it/s]
"blind schnorr": 0it [00:00, ?it/s]
"blind BLS": 0it [00:00, ?it/s]
"VOPRF": 100%|██████████| 1/1 [00:00<00:00,  1.14it/s]
"OPRF" "token": 0it [00:00, ?it/s]
"RSA blind signatures": 100%|██████████| 2/2 [00:00<?, ?it/s]
"anonymous tokens": 0it [00:00, ?it/s]
"private access token": 100%|██████████| 5/5 [00:04<00:00,  1.08it/s]
"trust token": 0it [00:00, ?it/s]
"private state token": 0it [00:00, ?it/s]
"unlinkable token": 0it [00:00, ?it/s]
"challenge bypass": 0it [00:00, ?it/s]
"blinded message": 0it [00:00, ?it/s]
"blind issuance": 0it [00:00, ?it/s]


GitLab: 10 relevante Repos gefunden, 500 gefiltert


In [4]:
# ── Mit GitHub-Ergebnissen vergleichen ───────────────────────────────────────

df_github = pd.read_excel(GITHUB_FILE)

# Vergleichsschlüssel: normalisierter Repo-Name + normalisierter Owner.
# Da viele Projekte auf GitHub UND GitLab gespiegelt werden (gleicher Name,
# teils gleicher Owner), prüfen wir zwei Stufen:
#   1. Exakter Match: owner/repo_name identisch
#   2. Name-Match: nur repo_name identisch (mögliche Mirrors mit anderem Owner)

def norm(s):
    return str(s).lower().strip().replace("-", "").replace("_", "")

github_full  = set(norm(o) + "/" + norm(r) for o, r in zip(df_github["owner"], df_github["repo_name"]))
github_names = set(norm(r) for r in df_github["repo_name"])

def overlap_status(row):
    full = norm(row["owner"]) + "/" + norm(row["repo_name"])
    if full in github_full:
        return "exact"          # identisches Projekt (Mirror)
    if norm(row["repo_name"]) in github_names:
        return "name_match"     # gleicher Name, anderer Owner → manuell prüfen
    return "unique"

df_gitlab["overlap"] = df_gitlab.apply(overlap_status, axis=1)

n_exact  = (df_gitlab["overlap"] == "exact").sum()
n_name   = (df_gitlab["overlap"] == "name_match").sum()
n_unique = (df_gitlab["overlap"] == "unique").sum()

print(f"Exakte Überschneidungen (Mirrors):  {n_exact}")
print(f"Namens-Übereinstimmungen (prüfen): {n_name}")
print(f"Nur auf GitLab:                     {n_unique}")

Exakte Überschneidungen (Mirrors):  2
Namens-Übereinstimmungen (prüfen): 0
Nur auf GitLab:                     8


In [5]:
# ── Nur nicht-überschneidende Repos exportieren ──────────────────────────────

# 'unique' sicher nicht auf GitHub; 'name_match' zur manuellen Prüfung mit ausgeben
df_out = df_gitlab[df_gitlab["overlap"].isin(["unique", "name_match"])].copy()
df_out = df_out.sort_values(["overlap", "stars"], ascending=[False, False]).reset_index(drop=True)

df_out.to_excel(OUTPUT_FILE, index=False)

print(f"Gespeichert: {OUTPUT_FILE} ({len(df_out)} Repos)")
print("\nHinweis: Repos mit overlap='name_match' haben denselben Namen wie ein")
print("GitHub-Repo, aber einen anderen Owner – kurz manuell prüfen ob Mirror oder")
print("eigenständiges Projekt.")

df_out[["overlap", "owner", "repo_name", "stars", "description"]].head(20)

Gespeichert: gitlab_only_repos_v2.xlsx (8 Repos)

Hinweis: Repos mit overlap='name_match' haben denselben Namen wie ein
GitHub-Repo, aber einen anderen Owner – kurz manuell prüfen ob Mirror oder
eigenständiges Projekt.


,overlap,owner,repo_name,stars,description
0,unique,poffey21,private-access-token-generator,6,
1,unique,brun0-matheus,ros-attacks-on-schnoor-blind-zero-knowledge-si...,1,
2,unique,johnturner,blindjoin,0,A standalone CoinJoin coordinator and client f...
3,unique,fumumue,PKP-Based_blind_signature,0,to make blind signature using fiat-shamir tran...
4,unique,Apeleg,ts-privacypass,0,Library for working with Private Access Tokens
5,unique,Vanaps,private-access-token-generator,0,
6,unique,mouradgustavo,private-access-token-generator,0,
7,unique,hartzell,private-access-token-generator,0,
